# 05_baseline.ipynb

Baseline: per-turn iterative inversion (Kim et al., 2026), no history.

Paths are relative: `data/` holds dialogues and embeddings (not included), `models/` holds fine-tuned weights (not included). See the README.


## Setup: models

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from nltk.translate.bleu_score import sentence_bleu
!pip install rouge
from rouge import Rouge
import json

device = "cuda"

llm_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForCausalLM.from_pretrained(
    llm_name,
    torch_dtype=torch.float32
).to(device)

llm.eval()

sent_model = SentenceTransformer("intfloat/e5-base-v2", device=device)
local_embed = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

## Scoring and candidate selection

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

lambda_reg = 0.1
tokenizer.pad_token = tokenizer.eos_token
llm.config.pad_token_id = tokenizer.pad_token_id

In [ ]:
def normalize(x):
    return (x - x.mean()) / (x.std() + 1e-8)

def compute_score(logprob, cos_sim, conf):
    return normalize(logprob) + conf * normalize(cos_sim)

def update_W(E, E_tilde):
    d = E.shape[1]
    I = torch.eye(d, device=device, dtype=torch.float32)
    return torch.linalg.inv(E.T @ E + lambda_reg * I) @ E.T @ E_tilde

def diversity_filter(token_embs, KS, T_hw):
    top_k = token_embs.size(0)
    selected_indices = []
    mask = torch.ones(top_k, dtype=torch.bool, device=token_embs.device)

    for i in range(top_k):
        if not mask[i]:
            continue

        selected_indices.append(i)
        if len(selected_indices) >= KS:
            break

        sims = F.cosine_similarity(token_embs[i].unsqueeze(0), token_embs, dim=1)
        sims[i] = 0
        mask = mask & (sims <= T_hw)
    return selected_indices

## Iterative generation

In [ ]:
gpu_total_mem = torch.cuda.get_device_properties(0).total_memory
threshold = 0.8 * gpu_total_mem
beam_size = 5

In [ ]:
def generate_text(reference_text, target_embedding, top_k=1000, KS=100, gamma=0.8, T_hw=0.9):
    beam = [" "]

    W_prev = None
    V_prev = None
    E_tilde_prev = None

    special_index = 1

    max_len = int(len(reference_text.split()) * 1.2)

    for t in range(max_len):

        inputs = tokenizer(beam, return_tensors="pt", padding=True).to(device)

        with torch.no_grad():
            outputs = llm(**inputs)

            lengths = inputs['attention_mask'].sum(dim=1)
            logits = outputs.logits[torch.arange(len(beam)), lengths-1, :]
            probs = torch.softmax(logits, dim=-1)
            topk_probs, topk_ids = torch.topk(probs, top_k, dim=-1)

        embedding_layer = llm.get_input_embeddings()
        token_embs = embedding_layer(topk_ids)

        candidates = []
        logprobs = []

        for b_idx, text in enumerate(beam):

            selected_indices = diversity_filter(token_embs[b_idx], KS, T_hw)

            for idx in selected_indices:
                token_id = topk_ids[b_idx, idx].item()
                logprobs.append(torch.log(topk_probs[b_idx, idx]))
                candidates.append(text + tokenizer.decode([token_id], skip_special_tokens=True))

        logprobs = torch.stack(logprobs)
        del inputs, outputs, logits, probs, token_embs, topk_probs, topk_ids

        E = torch.tensor(local_embed.encode(candidates), dtype=torch.float32, device=device)

        if W_prev is None:

            num_queries = int(KS * (gamma ** (t)))

            top50_idx = torch.topk(logprobs, num_queries).indices
            best_texts = [candidates[i] for i in top50_idx.tolist()]
            E_tilde = torch.tensor(
                sent_model.encode(best_texts),
                dtype=torch.float32,
                device=device
            )

            W = update_W(E[top50_idx], E_tilde)

            E_proj = E @ W
            cos_sims = F.cosine_similarity(E_tilde, target_embedding)

            E_proj_top50 = E[top50_idx] @ W
            cos_sims_tilde = F.cosine_similarity(E_proj_top50, E_tilde, dim=1)
            conf1 = 0.7 * cos_sims_tilde.mean()

            logprobs_50 = logprobs[top50_idx]
            scores = compute_score(logprobs_50, cos_sims, conf1)
            final_idx = torch.topk(scores, beam_size).indices.tolist()
            beam = [candidates[i] for i in final_idx]

            V_prev = top50_idx.tolist()
            W_prev = W
            W_prev_another = W
            E_tilde_prev = E_tilde
            special_index += 1

            del E, E_proj, cos_sims, scores
            continue

        E_proj = E @ W_prev
        cos_sims = F.cosine_similarity(E_proj, target_embedding)

        sims = []
        for idx, i in enumerate(V_prev):
            sims.append(
                F.cosine_similarity(
                    (E[i] @ W_prev).unsqueeze(0),
                    E_tilde_prev[idx].unsqueeze(0)
                )
            )
        conf_t = torch.mean(torch.stack(sims))

        scores_all = compute_score(logprobs, cos_sims, conf_t)

        sorted_idx = torch.argsort(scores_all)
        mid = len(sorted_idx) // 2

        num_queries = int(KS * (gamma ** (t)))
        best_group = sorted_idx[num_queries:]
        worst_group = sorted_idx[:num_queries]

        top50_group = best_group
        best_texts = [candidates[i] for i in top50_group.tolist()]
        E_tilde = sent_model.encode(best_texts, convert_to_tensor=True, device=device)
        E_best = E[top50_group]
        W = update_W(E_best, E_tilde)

        E_worst = E[worst_group]
        E_worst_proj = E_worst @ W

        combined_E = torch.cat([E_tilde, E_worst_proj], dim=0)
        combined_cos = F.cosine_similarity(combined_E, target_embedding)
        combined_logprob = torch.cat([logprobs[top50_group], logprobs[worst_group]])

        cos_sims_tilde1 = F.cosine_similarity(E_best @ W_prev_another, E_tilde, dim=1)
        conf2 = cos_sims_tilde1.mean()

        scores_final = compute_score(combined_logprob, combined_cos, conf2)
        combined_texts = best_texts + [candidates[i] for i in worst_group.tolist()]

        final_idx = torch.topk(scores_final, beam_size).indices.tolist()
        beam = [combined_texts[i] for i in final_idx]

        V_prev = top50_group.tolist()
        if special_index > 2:
          W_prev = W_prev_another
          W_prev_another = W
        else:
          W_prev_another = W
          special_index += 1
        E_tilde_prev = E_tilde

        del E, E_proj, E_tilde, W, cos_sims, scores_all
        del E_best, E_worst, E_worst_proj
        del combined_E, combined_cos, combined_logprob, scores_final
        mem_used = torch.cuda.memory_allocated()
        if mem_used > threshold:
          print(f"Step {t}: memory {mem_used/1e9:.2f} GB exceeds threshold, clearing cache")
          torch.cuda.empty_cache()

    return beam

## Text-quality evaluation

In [ ]:
from nltk.translate.bleu_score import SmoothingFunction
from sklearn.metrics.pairwise import cosine_similarity
from rouge import Rouge

smooth = SmoothingFunction().method1
rouge = Rouge()

In [ ]:
def evaluate_text(ref_texts, gen_texts):

    total = {"BLEU-1": 0, "BLEU-2": 0, "ROUGE-1": 0, "ROUGE-L": 0, "EmbeddingSim": 0}
    n = len(gen_texts)

    emb_gen = sent_model.encode(gen_texts, convert_to_numpy=True)
    emb_ref = sent_model.encode(ref_texts, convert_to_numpy=True)

    for i, (gen, ref) in enumerate(zip(gen_texts, ref_texts)):
        ref_tokens = ref.split()
        gen_tokens = gen.split()

        bleu1 = sentence_bleu([ref_tokens], gen_tokens, weights=(1, 0, 0, 0), smoothing_function=smooth)
        bleu2 = sentence_bleu([ref_tokens], gen_tokens, weights=(0.5, 0.5, 0, 0), smoothing_function=smooth)
        rouge_scores = rouge.get_scores(gen, ref)[0]
        sim = cosine_similarity([emb_gen[i]], [emb_ref[i]])[0][0]

        total["BLEU-1"] += bleu1
        total["BLEU-2"] += bleu2
        total["ROUGE-1"] += rouge_scores["rouge-1"]["f"]
        total["ROUGE-L"] += rouge_scores["rouge-l"]["f"]
        total["EmbeddingSim"] += sim

    avg_scores = {k: v / n for k, v in total.items()}
    return avg_scores

## Attack success and batch runner

In [ ]:
def evaluate_attack(gen_text, privacy, history, mode):
  if mode == 'u':
    name_flag = False
    name = privacy['name']
    prof = privacy['profession']
    prof_flag = False
    city = privacy['city']
    city_flag = False

    for text in gen_text:
      if name.lower() in text.lower():
        name_flag = True
      if prof.lower() in text.lower():
        prof_flag = True
      if city.lower() in text.lower():
        city_flag = True
    count = 0
    if name_flag:
      count += 1
      history['name'] += 1
    if prof_flag:
      count += 1
      history['profession'] += 1
    if city_flag:
      count += 1
      history['city'] += 1

    return count, history

  if mode == 'b':
    bill_flag = False
    bill = privacy['bill']
    income = privacy['income']
    income_flag = False
    investment = privacy['investment']
    investment_flag = False

    for text in gen_text:
      if str(bill) in text.lower() and 'bill' in text.lower():
        bill_flag = True
      if str(income) in text.lower() and 'income' in text.lower():
        income_flag = True
      if str(investment) in text.lower() and 'investment' in text.lower():
        investment_flag = True
    count = 0
    if bill_flag:
      count += 1
      history['bill'] += 1
    if income_flag:
      count += 1
      history['income'] += 1
    if investment_flag:
      count += 1
      history['flag'] += 1

    return count, history

  if mode == 'i':
    symptom = privacy['symptom']
    symptom_flag = False
    result = privacy['result']
    result_flag = False
    result_list = result.lower().split()
    solution = privacy['solution']
    solution_list = solution.lower().split()
    solution_flag = False

    for text in gen_text:
      if symptom.lower() in text.lower():
        symptom_flag = True
      if all(word in text.lower() for word in result_list):
        result_flag = True
      if all(word in text.lower() for word in solution_list):
        solution_flag = True
    count = 0
    if symptom_flag:
      count += 1
      history['symptom'] += 1
    if result_flag:
      count += 1
      history['result'] += 1
    if solution_flag:
      count += 1
      history['solution'] += 1

    return count, history

In [ ]:
def generate_batch(dataset, stop, mode):
  flag = True
  index_50 = 0
  ref_text = []
  gen_text = []
  num = 0
  attack = 0.0
  if mode == 'u':
    history = {'name': 0, 'profession': 0, 'city': 0}
  if mode == 'b':
    history = {'bill': 0, 'income': 0, 'investment': 0}
  if mode == 'i':
    history = {'symptom': 0, 'result': 0, 'solution': 0}

  for item in dataset[:stop]:
    if index_50 < 50:
      temp = []
    for d in item["dialogue"]:
      text = d["text"]
      ref_text.append(text)
      target_embedding = torch.tensor(d["embedding"]).to(device)

      new_text = generate_text(text, target_embedding)
      gen_text.append(new_text[0])
      if flag:
        print(f'Reference Text: {text}')
        print('Generated Texts of First Group:')
        for t in new_text:
          print(t)
        print('-'*200)

      if index_50 < 50:
        temp.append(new_text[0])
    flag = False
    if index_50 < 50:
      score, history = evaluate_attack(temp, item['privacy'], history, mode)
      num += score
    if index_50 == 49:
      attack = num / 150
      print(f"Success: {num}, Attack Success Rate: {attack}")
      print(f"History: {history}")
      print('-'*200)
    index_50 += 1
  print(f"Success: {num}, Attack Success Rate: {attack}")
  print(f"History: {history}")
  print('-'*200)
  return ref_text, gen_text

## Run: baseline

In [ ]:
# User-profile set: 50 groups
with open("data/new_user_profile_embed.json", "r") as f:
  user_data = json.load(f)

user_ref, user_gen = generate_batch(user_data, 50, 'u')
user_text_score = evaluate_text(user_ref, user_gen)
print(user_text_score)

In [ ]:
# Financial set: 73 groups (attack success is counted on the first 50)
with open("data/bank_embed.json", "r") as f:
  bank_data = json.load(f)

bank_ref, bank_gen = generate_batch(bank_data, 73, 'b')
bank_text_score = evaluate_text(bank_ref, bank_gen)
print(bank_text_score)

In [ ]:
# Medical set: 51 groups (attack success is counted on the first 50)
with open("data/ill_embed.json", "r") as f:
  ill_data = json.load(f)

ill_ref, ill_gen = generate_batch(ill_data, 51, 'i')
ill_text_score = evaluate_text(ill_ref, ill_gen)
print(ill_text_score)